# Navigator 에이전트 구축

앞선 실습에서 우리는 `Browser-use` 에이전트가 자연어 명령 하나만으로 브라우저를 직접 조종하는 것을 경험했습니다.
하지만 매번 브라우저를 새로 실행하고, 이전 대화를 전혀 기억하지 못한다면 실제 업무에서 쓰기엔 한계가 있겠죠?
이번 실습에서는 한 단계 더 나아가, **대화의 맥락을 기억하고 사용자와 자연스럽게 소통하는 'AI 웹 브라우징 어시스턴트'**, 즉 **Navigator 에이전트**를 직접 설계하고 구축합니다.

### 핵심 아이디어: Agent as Tool 패턴
이번 실습의 핵심은 **`Agent as Tool`** 패턴입니다. `Browser-use` 에이전트를 독립 실행 단위로 두는 것이 아니라, LangChain의 `@tool` 데코레이터로 감싸 **하나의 '도구(Tool)'로 전환**합니다.
그리고 이 도구를 더 똑똑한 상위 에이전트에게 쥐여주는 것입니다. 브라우저 조작이라는 '손발'과 대화·추론이라는 '머리'를 분리하는 것이 핵심입니다.


### 학습 흐름

| Phase | 내용 | 핵심 학습 |
|:---|:---|:---|
| **Phase 1** | 간단한 웹 어시스턴트를 직접 만들어보기 | Agent as Tool, 도구 반환값 설계, 세션 유지 |
| **Phase 2** | 실제 `app/`의 v1 Navigator 코드를 이식하며 이해하기 | 3단계 에스컬레이션, 전문 도구, 구조화 출력, 미들웨어 |

.env 파일을 로드하여 환경변수에 등록합니다. 노트북을 위한 실행 경로를 세팅합니다.

In [ ]:
import os, sys
from dotenv import load_dotenv

load_dotenv(override=True)

# PROJECT_ROOT를 .env에서 읽기 (없으면 현재 디렉토리)
project_root = os.getenv("PROJECT_ROOT", os.getcwd())

# 프로젝트 루트가 유효하지 않으면, 현재 위치에서 상위로 찾기
if not os.path.exists(os.path.join(project_root, "app")):
    # 상위 디렉토리 탐색
    current = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(current, "app")):
            project_root = current
            break
        current = os.path.dirname(current)

# Working Directory 설정
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import app

print(f"✅ Project Root: {project_root}")
print(f"✅ Working Directory: {os.getcwd()}")

In [ ]:
import asyncio
from browser_use import Agent, Browser, ChatGoogle

import nest_asyncio

# 주피터 노트북 환경에서 비동기 루프 중복 실행 오류를 방지하기 위해 사용합니다.
nest_asyncio.apply()

In [ ]:
os.environ["GOOGLE_API_KEY"]
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "develop_Navigator_agent" 

---
## Phase 1: 직접 만들어보기

### Part 1: 출발점 — 기본 웹 어시스턴트

02에서 배운 `Browser-use`를 LangChain `create_agent`의 **도구(Tool)**로 감싸면 어떤 에이전트가 만들어질까요?

핵심 아이디어는 **Agent as Tool** 패턴입니다:
- `Browser-use` 에이전트 = 브라우저를 조작하는 **"손발"**
- LangChain `create_agent` = 대화·추론하는 **"머리"**
- `@tool`로 감싸면 → 머리가 손발을 도구로 사용!

> 📖 LangChain의 `create_agent`와 `@tool` 문법은 `reference/LangChain/` 폴더에 정리되어 있습니다.

아래 코드에서 핵심 구성요소를 확인하세요:
1. **`@tool`** — Browser-use를 LangChain 도구로 래핑
2. **`Context`** — 런타임에 주입되는 상태 (일단 비어있는 상태로)
3. **`SYSTEM_PROMPT`** — 에이전트의 행동 지침
4. **`create_agent()`** — 모든 것을 조립하는 LangChain 에이전트 팩토리

In [ ]:
# ==========================================
# 1. 💡 런타임 공유 Context Data
# ==========================================
from dataclasses import dataclass
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from browser_use import Agent, ChatGoogle

@dataclass
class WebAssistantContext:
    # 사용자에 대한 기본 설정값 등을 넣을 수 있는 빈 컨텍스트로 둡니다.
    pass 

# ==========================================
# 2. 💡 범용 웹 탐색 도구 (Browser-use 기반)
# ==========================================
@tool
async def browse_web(instruction: str) -> str:
    """
    주어진 지시사항에 따라 웹 브라우저를 직접 조작하고 결과를 반환합니다.
    검색, 클릭, 텍스트 읽기, 웹사이트 분석 등 사용자가 요청한 웹 관련 작업을 수행할 때 사용하세요.
    
    Args:
        instruction: 브라우저가 수행해야 할 구체적인 행동 지시문 (예: '구글에서 2024년 올림픽 결과 검색해줘')
    """
    print(f"\n🌐 [Browser Tool] 행동 개시: {instruction}")
    
    bu_llm = ChatGoogle(model="gemini-flash-latest")
    
    # 전달받은 instruction을 그대로 Browser-use의 task로 전달합니다.
    agent = Agent(task=instruction, llm=bu_llm)
    history = await agent.run(max_steps=10)
    
    result_text = history.final_result()
    if not result_text:
        return "브라우저 조작을 시도했으나 명확한 결과를 얻지 못했습니다. 다른 명령으로 재시도해보세요."
    
    return result_text

# ==========================================
# 3. 💡 Web Assistant Agent (create_agent) 초기화
# ==========================================
# 멀티턴 대화를 위해 온도를 살짝 높여 자연스럽게 만듭니다 (0.1 -> 0.7)
model = init_chat_model("google_genai:gemini-flash-latest", temperature=0.7) 
store = InMemoryStore()
checkpointer = InMemorySaver()

SYSTEM_PROMPT = """
당신은 사용자와 대화하며 인터넷을 자유롭게 탐색하고 정보를 분석해주는 'AI 웹 브라우징 어시스턴트'입니다.

[역할 및 지침]
1. 당신은 `browse_web` 도구를 사용하여 스스로 웹 브라우저를 제어할 수 있습니다.
2. 사용자가 특정 사이트의 정보를 묻거나, 검색을 요청하거나, 웹 요약을 요구하면 적극적으로 도구를 사용하세요.
3. 도구를 사용할 때는 브라우저 에이전트(도구)가 명확하게 이해할 수 있도록 '구체적인 행동 명령(instruction)'을 작성하여 전달해야 합니다.
   - 나쁜 예: "네이버 뉴스 스크랩"
   - 좋은 예: "https://news.naver.com 에 접속해서 IT/과학 탭의 가장 위에 있는 기사 3개의 제목과 요약을 가져와"
4. 사용자와 자연스러운 멀티턴 대화를 이어나가며, 질문의 의도가 모호하다면 웹을 탐색하기 전에 되물어보세요.
5. 수집된 웹 탐색 결과를 단순히 전달하지 말고, 사용자의 문맥에 맞춰 요약하고 통찰력 있는 언어로 답변하세요.
"""

web_assistant_agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    context_schema=WebAssistantContext,
    tools=[browse_web],
    store=store,
    checkpointer=checkpointer)

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "my_browsing_session_1"}}
context = WebAssistantContext()

# 1번째 턴 (웹 탐색 유도)
query = "다나와 홈페이지에 접속해서 리뷰가 가장 많고 잘나가는 게이밍 마우스 상위 3개를 찾아봐."
print(f"👤 User: {query}")

response = await web_assistant_agent.ainvoke(
        {"messages": [HumanMessage(query)]},
        config=config,
        context=context
    )

print(f"\n🤖 Assistant: {response['messages'][-1].content}\n")
print("-" * 50)

지시대로 불명확한 지시의 경우 되묻거나, 직접 탐색을 수행합니다. 이어서 후속 질문을 해봅시다.

In [ ]:
# 2번째 턴
query = "가격과 이름을 알고 싶어."
print(f"👤 User: {query}")

response = await web_assistant_agent.ainvoke(
        {"messages": [HumanMessage(query)]},
        config=config,
        context=context
    )

print(f"\n🤖 Assistant: {response['messages'][-1].content}\n")
print("-" * 50)

In [ ]:
# 3번째 턴
query = "그 중 제일 싼 건 뭐야?"
print(f"👤 User: {query}")

response = await web_assistant_agent.ainvoke(
        {"messages": [HumanMessage(query)]},
        config=config,
        context=context
    )

print(f"\n🤖 Assistant: {response['messages'][-1].content}\n")
print("-" * 50)

In [ ]:
# 4번째 턴
query = "그 페이지에서 4번째로 인기있는 마우스의 이름은 뭔지 도구를 다시 쓰지 않고 답변할 수 있어?"
print(f"👤 User: {query}")

response = await web_assistant_agent.ainvoke(
        {"messages": [HumanMessage(query)]},
        config=config,
        context=context
    )

print(f"\n🤖 Assistant: {response['messages'][-1].content}\n")
print("-" * 50)

### Part 2: 도구 개선 — 세션 유지와 반환값 설계

앞선 멀티턴 대화 실험(4번째 턴)에서 한 가지 문제가 발견되었습니다.
*"그 페이지에서 4번째로 인기있는 마우스의 이름은 뭐야?"* 라고 물었을 때, 에이전트가 보고 있던 페이지에서 찾지 않고 **처음부터 다시 다나와를 검색**했습니다.

Navigator 에이전트(상위)는 대화 내용을 기억하고 있지만, `browse_web` 도구가 호출될 때마다 **완전히 새로운 브라우저 세션이 생성**되기 때문에 실제 브라우저는 이전 탐색 맥락(내가 어느 페이지에 있었는지)을 잃어버리는 것입니다.

이 문제를 해결하는 두 가지 방법을 차례대로 실습해 봅시다.

---

#### 🛠️ 방법 1: Tool의 반환값(return) 개선하기

에이전트가 도구의 동작 결과를 아는 **유일한 통로**는 도구의 **return 값**입니다.

기존 `browse_web`은 결과 텍스트만 반환했지만, **"마지막으로 방문한 URL"**을 함께 반환하면 어떨까요?
에이전트가 다음 턴에서 *"이 URL로 다시 접속해서..."* 라고 명령할 수 있게 됩니다.

> 💡 **핵심 인사이트**: 에이전트는 도구의 `return` 값만 봅니다. 도구가 반환하는 정보의 품질이 곧 에이전트의 판단 품질을 결정합니다.

In [ ]:
# ==========================================
# 실습 1: URL을 반환하는 새로운 도구 정의
# ==========================================
from browser_use import Agent, ChatGoogle

@tool
async def browse_web_url_mode(instruction: str) -> str:
    """
    주어진 지시사항에 따라 웹 브라우저를 조작하고, 결과와 함께 마지막 방문 URL을 반환합니다.

    Args:
        instruction: 브라우저가 수행해야 할 구체적인 행동 지시문
    """
    print(f"\n🌐 [Browser Tool v2] 행동 개시: {instruction}")
    
    bu_llm = ChatGoogle(model="gemini-flash-latest")
    agent = Agent(task=instruction, llm=bu_llm)
    history = await agent.run(max_steps=10)
    
    result_text = history.final_result()
    visited_urls = history.urls()
    last_url = visited_urls[-1] if visited_urls else "URL 정보 없음"
    
    if not result_text:
        return f"결과를 얻지 못했습니다.\n[마지막 방문 URL]: {last_url}"
    
    # ✅ 핵심: 결과 텍스트 + 마지막 방문 URL을 함께 반환
    return f"{result_text}\n\n[마지막 방문 URL]: {last_url}"


# 새로운 도구를 장착한 에이전트로 업데이트
web_assistant_agent_v2 = create_agent(
    model=model, system_prompt=SYSTEM_PROMPT, context_schema=WebAssistantContext,
    tools=[browse_web_url_mode], store=store, checkpointer=checkpointer
)

아래 코드에서 테스트해봅시다. URL을 받아서 어떻게 이어가는지 확인합니다.

In [ ]:
# 방법 1 테스트 
config_v2 = {"configurable": {"thread_id": "test_session_url"}}

# 1턴: 검색 지시
await web_assistant_agent_v2.ainvoke(
    {"messages": [HumanMessage("네이버 뉴스 IT/과학 메인 페이지에 들어가서 첫 번째 기사 제목을 알려줘.")]},
    config=config_v2
)

print("-" * 50)
# 2턴: 맥락 질문 (불필요한 첫 페이지 검색 없이 전달받은 URL로 직접 가는지 확인)
response = await web_assistant_agent_v2.ainvoke(
    {"messages": [HumanMessage("그 페이지에서 두 번째 기사 제목은 뭐야?")]},
    config=config_v2
)
print(f"\n🤖 Assistant: {response['messages'][-1].content}")

---

#### 🛠️ 방법 2: 하나의 브라우저 세션 유지하기 (keep_alive=True)

URL을 넘겨주는 방법은 페이지를 새로고침해야 하므로 스크롤 위치나 동적 상태(로그인 정보)가 초기화된다는 한계가 있습니다. 
가장 완벽한 해결책은 **하나의 브라우저를 열어두고 끄지 않는 것(keep_alive=True)**입니다. 
`browse_web` 도구들이 하나의 브라우저 객체를 닫지 않고 공유해서 사용하도록 만들어 봅시다.

In [ ]:
from browser_use import Browser
# 💡 핵심: 세션을 유지하는 공유 브라우저 인스턴스를 하나만 생성합니다.
shared_browser = Browser(
    headless=False,
    disable_security=True,
    window_size={'width': 1280, 'height': 720},
    keep_alive=True  # 도구 호출이 끝나도 브라우저가 종료되지 않습니다.
)

@tool
async def browse_web_keep_alive(instruction: str) -> str:
    """
    공유 브라우저 세션을 유지하며 웹을 탐색합니다.
    이전 탐색의 페이지, 스크롤 위치, 로그인 상태가 모두 보존됩니다.
    
    Args:
        instruction: 브라우저가 수행해야 할 구체적인 행동 지시문
    """
    print(f"\n🌐 [Browser Tool v3 - Keep Alive] 행동 개시: {instruction}")
    
    bu_llm = ChatGoogle(model="gemini-flash-latest")
    # ✅ 핵심: 매번 새 브라우저가 아닌, shared_browser를 재사용
    agent = Agent(task=instruction, llm=bu_llm, browser=shared_browser)
    history = await agent.run(max_steps=10)
    
    result_text = history.final_result()
    if not result_text:
        return "결과를 얻지 못했습니다."
    
    return result_text


# keep_alive 도구를 장착한 에이전트
web_assistant_agent_v3 = create_agent(
    model=model, system_prompt=SYSTEM_PROMPT, context_schema=WebAssistantContext,
    tools=[browse_web_keep_alive], store=store, checkpointer=checkpointer
)

In [ ]:
# 방법 2 테스트 
# 같은 페이지에서 탭을 끄지 않고 클릭/스크롤만으로 다음 목표를 찾는지 확인합니다.
config_v3 = {"configurable": {"thread_id": "test_session_keep_alive"}}

# 1턴: 검색 지시
await web_assistant_agent_v3.ainvoke(
    {"messages": [HumanMessage("위키백과에서 '인공지능'을 검색해서 개요 부분을 요약해줘.")]},
    config=config_v3
)

print("-" * 50)
# 2턴: 맥락 질문 (브라우저가 꺼지지 않고 현재 위키백과 페이지 안에서 이어서 탐색하는지 확인)
response = await web_assistant_agent_v3.ainvoke(
    {"messages": [HumanMessage("거기서 역사 섹션의 첫 문단만 그대로 읽어줄래?")]},
    config=config_v3
)
print(f"\n🤖 Assistant: {response['messages'][-1].content}")

# ⚠️ 실습이 끝난 후에는 수동으로 브라우저를 반드시 종료시켜주어야 합니다.
await shared_browser.stop()
print("\n🛑 공유 브라우저 세션 종료 완료")

#### 🎯 Phase 1 정리

| 항목 | 우리가 만든 에이전트 |
|:---|:---|
| 도구 | `browse_web` 1개 (Browser-use 래핑) |
| 프롬프트 | 5줄짜리 간단 지침 |
| 출력 | 자유 텍스트 |
| 세션 | keep_alive로 유지 |

이 에이전트는 **"범용 웹 어시스턴트"**로 충분히 유용하지만, **크롤링 전문가**로서는 한계가 있습니다:
- 매번 시각적 브라우저(Browser-use)를 띄우므로 **느리고 비용이 큼**
- HTML 구조를 직접 분석하지 못해 **CSS 셀렉터를 정밀하게 찾지 못함**
- 결과가 자유 텍스트이므로 **후속 에이전트(Coder)가 파싱하기 어려움**

> **Phase 2**에서는 `app/`에 구축된 실제 Navigator v1 코드를 살펴보며, 이 한계를 어떻게 극복했는지 이해합니다.

---
## Phase 2: v1 코드 이식 — 실제 Navigator의 내부 구조

### Part 3: Navigator의 핵심 전략 — 3단계 에스컬레이션 사다리

크롤링 전문가는 처음부터 비싼 브라우저를 띄우지 않습니다. **가벼운 방법부터 시도하고, 실패할 때만 한 단계씩 올라갑니다.**

```
┌─ Level 1: HTML 분석 (기본) ──────────────────────┐
│  get_page_structure → verify_selectors             │
│  빠르고 저렴. 정적/SSR 페이지는 이것만으로 충분       │
└──────────────────────────────────────────────────┘
        ↓ (셀렉터가 빈 배열, 데이터가 JS 렌더링인 경우)
┌─ Level 2: 하이브리드 (핵심 전략) ───────────────────┐
│  browse_web으로 "딱 하나의 마이크로 액션"만 수행         │
│  → 변화된 URL을 get_page_structure로 다시 분석         │
│  ★ 이것이 가장 효율적인 전략입니다                      │
└──────────────────────────────────────────────────┘
        ↓ (URL이 바뀌지 않거나 JS-only 렌더링인 경우)
┌─ Level 3: 풀 브라우저 (최후의 보루) ────────────────┐
│  browse_web에게 "셀렉터 탐색 + 데이터 확인"을 위임    │
└──────────────────────────────────────────────────┘
```

Phase 1에서 만든 에이전트는 **항상 Level 3**만 사용했습니다.
이제 Level 1, 2를 추가하여 **비용과 정확도를 모두 개선**해 봅시다.

이 전략을 구현하려면 **3개의 전문 도구**가 필요합니다:
1. **`get_page_structure`** — HTML을 분석하여 CSS 셀렉터 후보를 찾는 도구 (Level 1)
2. **`verify_selectors_with_samples`** — 찾은 셀렉터가 실제 데이터를 가져오는지 검증하는 도구 (Level 1)
3. **`browse_web`** — 동적 인터랙션이 필요할 때 실제 브라우저를 제어하는 도구 (Level 2~3)

### Part 4: 도구(Tools) 설계

#### 📂 `app/tools/navigator.py`

Navigator는 3개의 전문 도구를 사용합니다. LangChain의 `@tool(parse_docstring=True)` 데코레이터를 사용하면, Docstring에 작성한 설명과 인자 정보가 **LLM이 이해하는 도구 스키마로 자동 변환**됩니다.

> 📖 `@tool` 선언 문법은 `reference/LangChain/tool_declaration.md`를 참고하세요.

---

#### 도구 1: `get_page_structure` — HTML 정적 분석 (Level 1)

브라우저를 시각적으로 띄우지 않고 **백그라운드에서 HTML을 수집**하여, 내부 LLM(gemini-flash)이 CSS 셀렉터 후보를 찾아냅니다. 가장 빠르고 토큰 비용이 저렴한 **주력 분석 도구**입니다.

동작 원리:
1. `crawl4ai`로 페이지 HTML 수집 (headless 브라우저)
2. `BeautifulSoup`으로 불필요한 태그(script, style 등) 제거
3. 정제된 HTML을 `gemini-flash`에 전달하여 CSS 셀렉터 분석
4. JSON 형태로 셀렉터 결과 반환

In [ ]:
# 📂 이 코드는 app/tools/navigator.py에 있습니다.
import json
import re
from langchain.tools import tool
from bs4 import BeautifulSoup
from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig, CacheMode
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

@tool(parse_docstring=True)
async def get_page_structure(url: str, scraping_goal: str, wait_seconds: float = 2.0) -> str:
    """웹페이지 HTML을 내부 LLM이 직접 분석하여 CSS 셀렉터 결과만 반환합니다.
    
    Args:
        url: 분석할 웹페이지 URL
        scraping_goal: 수집하려는 데이터 설명
        wait_seconds: 페이지 로드 후 AJAX 대기 시간(초). 정적 사이트는 기본값(2초), AJAX가 많은 동적 사이트는 5~8초를 권장합니다.
    """
    print(f"\n📐 [get_page_structure] {url} (Goal: {scraping_goal}, Wait: {wait_seconds}s)")
    browser_cfg = BrowserConfig(headless=True, java_script_enabled=True)
    run_cfg = CrawlerRunConfig(cache_mode=CacheMode.BYPASS, page_timeout=30000, delay_before_return_html=wait_seconds)

    try:
        async with AsyncWebCrawler(config=browser_cfg) as crawler:
            result = await crawler.arun(url=url, config=run_cfg)
    except Exception as e:
        return f"[Error] HTML 수집 실패: {e}\n→ browse_web을 사용하세요."

    raw_html = result.html or ""
    soup = BeautifulSoup(raw_html, "html.parser")
    for tag in soup(["script", "style", "noscript", "svg", "path", "header", "footer"]):
        tag.decompose()
        
    structured_html = soup.prettify()
    if not structured_html.strip():
        return "[Warning] HTML이 비어 있습니다. JS 로드 실패 가능성."

    structured_html = structured_html[:300000] # 토큰 제한
    analysis_llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0)
    
    prompt = f"""
    아래 HTML에서 "{scraping_goal}" 요소를 찾아 CSS 셀렉터를 반환하세요. 응답은 JSON만 출력.
    
    [주의사항]
    - 광고/스폰서 영역(PowerShopping, ad, sponsored 등)의 셀렉터는 절대 반환하지 마세요.
    - bridge, redirect, tracking, affiliate 등이 포함된 URL을 href로 가진 링크는 광고 링크이므로 무시하세요.
    - 실제 콘텐츠 영역(main, article, product list 등)에서만 셀렉터를 찾으세요.
    
    [HTML] {structured_html}
    [구조]
    {{"selectors": {{"필드": "셀렉터"}}, "samples": {{...}}, "navigate_to_next": "셀렉터 또는 null", "pagination": "방식", "confidence": "high/low"}}
    """
    resp = await analysis_llm.ainvoke([HumanMessage(prompt)])
    content = resp.content
    if isinstance(content, list): content = "".join([c.get("text", "") if isinstance(c, dict) else str(c) for c in content])
    
    json_match = re.search(r'\{.*\}', content, re.DOTALL)
    if json_match:
        return json.dumps(json.loads(json_match.group()), ensure_ascii=False, indent=2)
    return content

In [ ]:
# 🧪 독립 실행 테스트: quotes.toscrape.com 분석
result = await get_page_structure.ainvoke({
    "url": "http://quotes.toscrape.com",
    "scraping_goal": "명언(quote), 저자(author), 태그(tags)"
})
print(result)

---

#### 도구 2: `verify_selectors_with_samples` — 셀렉터 검증 (Level 1)

`get_page_structure`가 찾아낸 CSS 셀렉터 후보가 **실제로 유효한지 검증**하는 도구입니다.
실제 Playwright 브라우저를 띄워 셀렉터를 적용하고, **최대 5개의 실제 데이터 샘플**을 반환합니다.

- 샘플 데이터 배열이 비어있으면(`[]`) → 그 셀렉터는 실패한 것
- `::attr(href)` 같은 속성 셀렉터도 지원

In [ ]:
# 📂 이 코드는 app/tools/navigator.py에 있습니다.
from playwright.async_api import async_playwright

@tool(parse_docstring=True)
async def verify_selectors_with_samples(url: str, selectors_json: str) -> str:
    """찾아낸 CSS 셀렉터가 실제로 데이터를 가져오는지 검증합니다.
    
    Args:
        url: 검증할 웹페이지 URL
        selectors_json: JSON 문자열로 된 셀렉터 딕셔너리 예) '{"title": "a.title"}'
    """
    print(f"\n🔍 [verify_selectors] {url}")
    try: sel_dict = json.loads(selectors_json)
    except: return "[Error] 유효한 JSON 문자열 포맷이어야 합니다."
    
    results = {k: [] for k in sel_dict.keys()}
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            page = await browser.new_page()
            await page.goto(url, wait_until="domcontentloaded", timeout=15000)
            await page.wait_for_timeout(2000)
            
            for k, sel in sel_dict.items():
                act_sel = sel
                attr_name = ""
                if "::attr(" in sel:
                    match = re.search(r'(.*?)::attr\((.*?)\)', sel)
                    if match:
                        act_sel, attr_name = match.group(1).strip(), match.group(2).strip()
                elements = await page.query_selector_all(act_sel)
                for el in elements[:5]:
                    val = await el.get_attribute(attr_name) if attr_name else await el.text_content()
                    if val: results[k].append(val.strip())
            await browser.close()
            
            out = []
            for k, v in results.items():
                out.append(f"[✅ OK] {k}: 샘플 {v}" if len(v) > 0 else f"[⚠️ FAILED] {k}: 매칭 0건")
            return "\\n".join(out)
    except Exception as e:
        return f"[Error] 브라우저 검증 중 오류: {e}" 

In [ ]:
# 🧪 독립 실행 테스트: get_page_structure 결과를 검증
selectors = json.dumps({"quote": "span.text", "author": "small.author", "tags": "div.tags a.tag"})
result = await verify_selectors_with_samples.ainvoke({
    "url": "http://quotes.toscrape.com",
    "selectors_json": selectors
})
print(result)

---

#### 도구 3: `browse_web` — 브라우저 제어 (Level 2~3)

동적 인터랙션(클릭, 스크롤, 필터)이 필요할 때 실제 브라우저를 제어하는 도구입니다.

Phase 1의 `browse_web`과 비교하면 크게 3가지가 다릅니다:

| | Phase 1 browse_web | v1 browse_web |
|:---|:---|:---|
| **인자** | `instruction` 1개 | `url`, `instruction`, `purpose`, `context` 4개 |
| **브라우저** | 매번 새로 생성 | `ToolRuntime`으로 **공유 브라우저** 주입 |
| **하위 에이전트 지침** | 없음 | `BROWSER_AGENT_GUIDE` 주입 |

> 📖 **`ToolRuntime[Context]`** — 에이전트 런타임의 상태(Context, Store)를 도구 내부에서 접근하는 패턴입니다.
> LLM에게는 보이지 않으며, 프레임워크가 실행 시 자동으로 주입합니다.
> 자세한 내용은 `reference/LangChain/runtime.md`를 참고하세요.

`browse_web`은 두 가지 외부 모듈에 의존합니다:
- **`NavigatorContext`** (Part 6에서 자세히 설명) — 공유 브라우저 인스턴스 주입용
- **`BROWSER_AGENT_GUIDE`** (Part 5에서 자세히 설명) — 하위 에이전트 행동 지침

여기서는 `app/` 패키지에서 임포트하여 사용합니다.

In [ ]:
# 📂 이 코드는 app/tools/navigator.py에 있습니다.
# NavigatorContext와 BROWSER_AGENT_GUIDE는 각각 Part 6, Part 5에서 자세히 살펴봅니다.
from langchain.tools import tool, ToolRuntime
from browser_use import Agent, Browser, ChatOpenAI
from app.schemas import NavigatorContext
from app.prompts.navigator import BROWSER_AGENT_GUIDE
from app.tools.common import CostTracker

_cost_tracker = CostTracker()

@tool(parse_docstring=True)
async def browse_web(runtime: ToolRuntime[NavigatorContext], url: str, instruction: str, purpose: str = "", context: str = "") -> str:
    """동적 인터랙션 및 시각적 검증이 필요할 때 실제 브라우저로 웹을 제어합니다.
    
    Args:
        url: 이동할 URL (이어서 작업하려면 빈 문자열)
        instruction: 수행할 단일 마이크로 액션 (예: '더보기 버튼 클릭 후 변경된 URL 반환')
        purpose: 이 액션의 목적 (예: '쇼핑몰 리스트를 전부 펼치기 위해')
        context: Navigator가 파악한 현재까지의 전략적 상황
    """
    # Navigator가 전달한 전략적 컨텍스트 조합
    task_parts = [BROWSER_AGENT_GUIDE]
    if purpose:
        task_parts.append(f"[이 작업의 목적]\n{purpose}")
    if context:
        task_parts.append(f"[현재 상황 (상위 에이전트 제공)]\n{context}")
    if url:
        task_parts.append(f"[작업]\nNavigate to '{url}' and perform this task: {instruction}")
    else:
        task_parts.append(f"[작업]\nPerform this task on the current page: {instruction}")
    
    task = "\n\n".join(task_parts)

    llm = ChatOpenAI(model="gpt-4.1-mini")
    user_browser = getattr(runtime.context, "shared_browser", None)
    
    if user_browser:
        agent = Agent(task=task, llm=llm, browser=user_browser, calculate_cost=True)
        history = await agent.run(max_steps=15)
    else:
        tb = Browser(headless=True)
        agent = Agent(task=task, llm=llm, browser=tb, calculate_cost=True)
        history = await agent.run(max_steps=15)
        await tb.stop()
    
    # 비용 기록
    _cost_tracker.record_usage(instruction[:50], history.usage)
    
    return history.final_result() or "결과 반환 없음" 

---
### Part 5: 프롬프트(Prompts) — Navigator의 두뇌 설계

#### 📂 `app/prompts/navigator.py`

도구만으로는 부족합니다. Navigator가 **언제, 어떤 순서로, 어떤 도구를** 사용해야 하는지를 판단하는 것은 **시스템 프롬프트**의 역할입니다.

`NAVIGATOR_SYSTEM_PROMPT`는 다음 5가지 섹션으로 구성되어 있습니다:

| 섹션 | 역할 |
|:---|:---|
| **역할 정의** | "크롤링 파이프라인의 총괄 매니저이자 아키텍트" |
| **에스컬레이션 사다리** | Level 1 → 2 → 3 순서로 도구를 사용하라는 규칙 |
| **도구별 상세 가이드** | wait_seconds 판단 기준, 마이크로 액션 원칙 등 |
| **Blueprint 판단 가이드** | rendering_type, pagination_method 등의 판단 기준 |
| **핵심 원칙** | "Coder는 HTML을 전혀 볼 수 없다!" |

> 💡 **프롬프트는 에이전트의 행동을 결정하는 가장 강력한 레버**입니다. 같은 도구를 가지고도 프롬프트에 따라 성능이 극적으로 달라집니다.

In [ ]:
# 📂 이 코드는 app/prompts/navigator.py에 있습니다.
NAVIGATOR_SYSTEM_PROMPT = """
당신은 웹 크롤링 파이프라인의 총괄 매니저이자 아키텍트인 'Navigator'입니다.
도구를 상황에 맞게 유연하게 사용하고,
최종적으로 Coder가 즉시 실행할 수 있는 안정적이고 정교한 크롤링 Blueprint를 설계합니다.

당신의 역할은 도구를 기계적으로 실행하는 것이 아니라,
웹 구조를 분석하고 판단하여 데이터 수집 전략(청사진)을 치밀하게 세우는 것입니다.

════════════════════════════════════════
[핵심 원칙: 3단계 에스컬레이션 사다리]
════════════════════════════════════════

도구는 반드시 아래 순서대로 시도하세요. 가벼운 도구부터 시작하고,
실패할 때만 다음 단계로 올라갑니다. 절대로 건너뛰지 마세요.

  ┌─ Level 1: HTML 분석 (기본) ──────────────────────┐
  │  get_page_structure → verify_selectors_with_samples │
  │  빠르고 저렴. 정적/SSR 페이지는 이것만으로 충분.       │
  └──────────────────────────────────────────────────┘
          ↓ (셀렉터가 빈 배열, 데이터가 JS 렌더링인 경우)
  ┌─ Level 2: 하이브리드 (핵심 전략) ───────────────────┐
  │  browse_web으로 "딱 하나의 마이크로 액션"만 수행         │
  │  (예: '더보기 클릭', '필터 하나 체크', '탭 전환')        │
  │  → 변화된 URL 또는 상태를 받아옴                       │
  │  → 그 URL을 get_page_structure로 다시 분석             │
  │                                                       │
  │  ★ 이것이 가장 효율적인 전략입니다.                     │
  │  browse_web에게 복잡한 멀티스텝 작업을 맡기지 마세요!   │
  │  한 번에 하나의 행동만 시킨 뒤, 결과를 HTML로 분석하세요.│
  └──────────────────────────────────────────────────┘
          ↓ (URL이 바뀌지 않거나 JS-only 렌더링인 경우)
  ┌─ Level 3: 풀 브라우저 (최후의 보루) ────────────────┐
  │  browse_web에게 "셀렉터 탐색 + 데이터 확인"을 위임    │
  │  이 경우에도 instruction을 매우 구체적으로 작성하세요.  │
  └──────────────────────────────────────────────────┘

════════════════════════════════════════
[도구 역할 및 사용 전략]
════════════════════════════════════════

■ get_page_structure(url, scraping_goal, wait_seconds=2.0)
  - 가장 빠르고 토큰 비용이 저렴한 주력 분석 도구입니다.
  - 브라우저를 시각적으로 띄우지 않고 백그라운드에서 HTML 전체를 분석하여 CSS 셀렉터 후보를 찾아냅니다.
  - [wait_seconds 판단 기준]:
    정적(SSR) 사이트 → wait_seconds=2 (기본값, 빠르게 분석)
    가벼운 AJAX 사이트 → wait_seconds=4~5
    무거운 AJAX 사이트(다나와, 쿠팡 등 대형 쇼핑몰) → wait_seconds=7~8
    판단법: URL에 접속했을 때 상품 목록이 JS로 동적 로드되는 사이트라면 wait_seconds를 높이세요.
    만약 confidence가 "low"로 나오면, wait_seconds를 높여서 재시도하세요.

■ verify_selectors_with_samples(url, selectors_json)
  - [필수 사용] get_page_structure가 찾아낸 셀렉터 후보가 실제로 유효한지 검증하는 강력한 도구입니다.
  - 이 도구는 실제 브라우저를 띄워 입력받은 CSS 셀렉터를 즉시 적용해보고 최대 5개의 실제 추출된 리얼 데이터를 반환합니다.
  - 샘플 데이터 배열이 비어있거나([]), "None" 이거나, 잘못된 값이라면 그 셀렉터는 실패한 것입니다. 즉시 셀렉터를 수정하여 다시 검증하거나 다른 도구를 사용해야 합니다.

■ browse_web(runtime, url, instruction)
  - 시각적 검증과 동적 행동(클릭, 스크롤, 대기)이 필요할 때 사용합니다.
  - [중요: 마이크로 액션 원칙]
    browse_web에게 "필터 3개 클릭하고, 정렬하고, 결과 URL 뽑아줘"처럼
    복잡한 멀티스텝 작업을 한꺼번에 맡기면 높은 확률로 실패합니다.
    대신 이렇게 사용하세요:
      → browse_web(url, "32GB 필터를 클릭하고 변경된 URL을 알려줘")
      → 반환된 URL을 get_page_structure(new_url, ...)로 다시 분석
      → 필요하면 browse_web(new_url, "다음 필터 클릭...")으로 한 단계 더 진행
    이렇게 한 번에 하나의 액션만 위임하면 실패율이 극적으로 낮아집니다.

■ 도구 실패 시 전환 규칙:
  - get_page_structure가 "[Error]" 또는 "[Warning]"으로 시작하는 응답을 반환하면,
    같은 URL로 재시도하지 말고 즉시 browse_web으로 전환하세요.
  - get_page_structure가 HTML을 수집했지만 셀렉터를 제대로 찾지 못한 경우에만
    scraping_goal을 더 구체적으로 바꿔서 재시도 하세요.

════════════════════════════════════════
[Blueprint 핵심 판단 가이드: Coder에게 넘겨줄 필수 정보]
════════════════════════════════════════
**아래 항목들은 Coder가 코드를 짜는 핵심 기준이 되므로 매우 정확하게 판단해야 합니다.**

1. rendering_type (렌더링 방식)
   - "Static SSR": URL 접속 즉시 HTML 원본에 데이터가 정적으로 포함된 경우. (BeautifulSoup 활용)
   - "Dynamic CSR/JS": 상호작용이나 대기 후에야 자바스크립트로 데이터가 채워지는 경우. (Playwright 활용)

2. pagination_method (페이지 이동 방식)
   - "URL파라미터": 2페이지 이동 시 ?page=2 처럼 URL이 변경됨
   - "AJAX버튼": URL 변경 없이 '더보기' 버튼 등으로 목록이 추가됨
   - "무한스크롤": 마우스 스크롤을 내리면 자동 로드됨
   - "None": 페이징 없음
   판단 기준:
   - URL에 ?page=, &p=, /page/ 등이 있으면 → "URL파라미터"
   - "더보기", "Load More" 버튼이 보이면 → "AJAX버튼"
   - 확인이 안 되면, get_page_structure로 하단 영역의 페이지네이션 요소를 별도로 분석하세요.

3. 셀렉터 정밀 검증 (Crucial):
   - 도구(get_page_structure 또는 browse_web)가 셀렉터 후보를 알려주면, 그게 진짜 "요소 1개"를 뜻하는지, "반복되는 컨테이너"를 뜻하는지 구분하세요.
   - 단순히 `a.sa_text_title` 라고만 적으면 Coder가 이게 텍스트인지 링크인지 헷갈립니다.
   - 반드시 텍스트 추출용 셀렉터(예: `title: "a.sa_text_title"`)와 링크 추출용 속성 셀렉터(예: `link: "a.sa_text_title::attr(href)"`)를 명확하게 분리해서 Blueprint에 적으세요.
   - 컨테이너가 존재한다면 (예: 루프를 돌아야 하는 `div.sa_text`), 부모 컨테이너 셀렉터를 별도로 명시하는 것이 가장 좋습니다.

4. anti_bot_notes (장애 요소 및 주의사항)
   - 로그인 창으로 튕기는지, 캡차가 뜨는지, 동적 팝업이 뜨는지 상세하게 적어줍니다.
   - 사이트 특유의 방해 요소(예: 다나와의 용어사전 팝업)도 반드시 기록하세요.

════════════════════════════════════════
[매우 중요한 원칙]
════════════════════════════════════════
- 당신 뒤에 있는 Coder 에이전트는 웹사이트의 HTML이나 화면을 전혀 볼 수 없는 상태입니다. 단지 당신의 Blueprint 정보에만 의존합니다.
- 애매한 값으로 대충 넘기면 파이프라인은 100% 실패합니다.
- 만약 사용자가 "안녕하세요"와 같이 단순 인사를 하거나 구체적인 크롤링 URL/Goal 지시가 없는 상태라면, 웹탐색 도구를 실행하지 말고 자연어로 친절하게 답변하세요.
- 확신이 들 때까지 도구를 사용해 검증하고 꼼꼼하게 작성하세요.
"""

print(f"✅ NAVIGATOR_SYSTEM_PROMPT 로드 완료 ({len(NAVIGATOR_SYSTEM_PROMPT)}자)")

---

#### `BROWSER_AGENT_GUIDE` — 하위 에이전트(browse_web 내부) 행동 지침

`browse_web` 도구 내부에서 실행되는 Browser-use 에이전트에게 주입되는 **행동 원칙**입니다.
스크래핑 작업에서의 효율성을 극대화하기 위한 6가지 규칙이 담겨 있습니다.

In [ ]:
# 📂 이 코드는 app/prompts/navigator.py에 있습니다.
BROWSER_AGENT_GUIDE = """[당신의 역할]
당신은 데이터 스크래핑 파이프라인의 일부로, 상위 에이전트(Navigator)의 지시를 받아 브라우저를 조작합니다.
스크래핑 작업에서는 속도와 정확성이 핵심이므로, 아래 원칙을 따르세요.

[효율적 행동 원칙]
1. evaluate는 단독 사용: evaluate()는 다른 액션과 같은 스텝에 넣지 마세요. evaluate는 시퀀스를 종료시켜 뒤의 액션이 취소됩니다.
   → URL 확인이 필요하면 evaluate만 단독으로 한 스텝에 실행하고, 다음 스텝에서 find_elements를 하세요.
   → URL 확인은 최초 1회만 하세요. 이미 확인한 URL을 반복 확인하지 마세요.
2. 읽기 > 클릭: URL이나 텍스트를 알아내야 할 때, 클릭하여 이동하는 대신 find_elements로 href/textContent를 직접 읽으세요.
   → 이유: 클릭은 페이지 전환, 팝업, 리다이렉트 등 예측 불가능한 부작용을 유발합니다.
3. 반복 금지: 같은 액션(같은 셀렉터, 같은 evaluate 코드)을 2번 이상 실행하지 마세요. 결과가 같으면 즉시 다른 접근법을 시도하세요.
   → 이유: 동일 행동의 반복은 스텝 예산을 소진시키고 루프에 빠지게 합니다.
4. 광고/트래킹 URL 구별: bridge, redirect, tracking, ad, affiliate 등의 키워드가 포함된 URL은 광고 리다이렉트일 가능성이 높습니다. 이런 URL은 무시하고 사이트 본래의 상세 페이지 URL 패턴을 찾으세요.
   → 이유: 광고 URL을 따라가면 Access Denied 등의 장벽에 막혀 스텝이 낭비됩니다.
5. DOM 구조 분석: 셀렉터를 찾아야 할 때는 find_elements보다 evaluate()로 JavaScript를 실행하여 DOM 트리를 직접 읽으세요.
   → 이유: find_elements는 매칭 갯수만 반환하고 DOM 계층 구조는 보여주지 않습니다.
6. 빠른 실패 보고: 페이지 내용이 요청과 맞지 않으면 (예: 키보드를 찾는데 의류 페이지가 나옴) 즉시 done(text="[FAIL] 이유: ...", success=False)로 실패를 보고하세요. 3스텝 이내에 원하는 데이터를 찾지 못하면 더 이상 시도하지 말고 실패를 보고하세요.
   → 이유: 잘못된 페이지에서 12스텝을 소진하면 전체 파이프라인의 예산이 낭비됩니다.
"""

print(f"✅ BROWSER_AGENT_GUIDE 로드 완료 ({len(BROWSER_AGENT_GUIDE)}자)")

---
### Part 6: 스키마(Schemas) + 구조화 출력 — Blueprint 설계

#### 📂 `app/schemas.py`

Phase 1에서 Navigator의 응답은 **자유 텍스트**였습니다. 하지만 실제 파이프라인에서는 Navigator의 출력을 **Coder 에이전트가 파싱**해야 합니다.

**왜 구조화된 출력(Blueprint)이 필요한가?**
- Coder가 JSON 키를 직접 읽어 코드를 생성 → 파싱 오류 없음
- 필수 필드가 빠지면 Pydantic 검증에서 즉시 에러 → 빠른 진단
- 파이프라인의 **안정성과 재현성**이 극적으로 향상

**ToolStrategy 개념:**
LangChain의 `ToolStrategy(OutputSchema)`를 `response_format`으로 지정하면, 에이전트가 **도구를 다 사용한 후 최종 응답을 반드시 이 스키마 형태로** 출력하도록 강제합니다.

```python
agent = create_agent(
    ...
    response_format=ToolStrategy(NavigatorBlueprintCollection)  # ← 이것!
)
```

In [ ]:
# 📂 이 코드는 app/schemas.py에 있습니다.
import json
from typing import Optional, Any
from pydantic import BaseModel, Field, field_validator
from dataclasses import dataclass, field

# ==========================================
# 1. 멀티에이전트 Blueprint 파서 데이터 모델
# ==========================================
class PageLayer(BaseModel):
    """하나의 탐색 계층을 표현하는 단위 블록"""
    layer_name: str = Field(description="이 계층의 역할 이름 (예: '기사 목록', '상품 상세')")
    url_pattern: str = Field(description="이 계층의 URL 구조 예시 또는 진입점 URL")
    container_selector: Optional[str] = Field(default=None, description="반복 항목의 부모 컨테이너 CSS 셀렉터 (예: 'table.mall_list tbody tr'). Coder는 이 셀렉터로 루프를 돌고, selectors의 각 필드를 하위에서 찾습니다.")
    selectors: dict[str, str] = Field(description="이 계층에서 수집할 데이터의 CSS 셀렉터 딕셔너리")
    pre_actions: Optional[list[str]] = Field(default=None, description="데이터 로드 전 수행해야 할 브라우저 액션 목록 (예: ['더보기 버튼(.btn_list_more) 클릭', '탭 전환: 쇼핑몰별 최저가']). 순서대로 실행됩니다.")
    navigate_to_next: Optional[str] = Field(default=None, description="다음 계층으로 이동하는 링크의 CSS 셀렉터. 마지막 계층이면 무조건 None.")
    pagination_method: Optional[str] = Field(default=None, description="페이지네이션 방식 (URL파라미터 / AJAX버튼 / 무한스크롤 / None)")

    @field_validator("selectors", mode="before")
    @classmethod
    def parse_selectors(cls, v):
        if isinstance(v, str):
            try: return json.loads(v)
            except Exception: pass
        return v

    @field_validator("navigate_to_next", "pagination_method", mode="before")
    @classmethod
    def parse_none_string(cls, v):
        if v in ("None", "null", "없음", "N/A", ""): return None
        return v

class NavigatorBlueprint(BaseModel):
    """Navigator가 Coder에게 전달하는 동적 N계층 크롤링 설계 도면"""
    entry_urls: list[str] = Field(description="크롤링을 시작할 URL 목록 (구조가 같다면 복수 가능)")
    total_layers: int = Field(description="탐색에 필요한 총 계층 수")
    layers: list[PageLayer] = Field(description="탐색 순서대로 정렬된 PageLayer 목록")
    rendering_type: str = Field(description="Static SSR 또는 Dynamic CSR/JS")
    anti_bot_notes: str = Field(description="로그인, 캡차 등 안티봇 주의사항. 제약이 없으면 '없음'")

class NavigatorBlueprintCollection(BaseModel):
    """Navigator가 반환하는 Blueprint 모음"""
    total_jobs: int = Field(description="총 Blueprint 구조체 수")
    blueprints: list[NavigatorBlueprint] = Field(description="수집 작업별 Blueprint 목록")

print("✅ Blueprint 스키마 정의 완료: PageLayer → NavigatorBlueprint → NavigatorBlueprintCollection")

#### `NavigatorContext` — 런타임 상태 (Context)

에이전트 실행 시 주입되는 런타임 상태입니다. 두 가지 역할을 합니다:
- **`shared_browser`**: 공유 브라우저 인스턴스 (Part 2의 keep_alive와 같은 원리!)
- **`response_mode`**: `"chat"` 또는 `"blueprint"` — 응답 포맷을 동적으로 전환

In [ ]:
# 📂 이 코드는 app/schemas.py에 있습니다.
@dataclass
class NavigatorContext:
    shared_browser: Optional[Any] = None  # Browser 인스턴스 주입용
    response_mode: str = field(default="chat") # "chat" or "blueprint"

print("✅ NavigatorContext 정의 완료")

---
### Part 7: 미들웨어(Middleware) — 런타임 동작 제어

#### 📂 `app/agents/utils.py`

**"채팅할 때는 Blueprint 말고 자연어로!"**

Navigator에게 `ToolStrategy(NavigatorBlueprintCollection)`을 걸어두면, 에이전트는 **항상** Blueprint JSON 형태로 응답합니다.
하지만 서빙 시 사용자와 대화할 때는 자연어로 답변해야 하죠.

이 문제를 해결하는 것이 **미들웨어(Middleware)**입니다.

> 📖 미들웨어에 대한 자세한 설명은 `reference/LangChain/middleware.md`를 참고하세요.

**`@wrap_model_call`** 패턴을 사용하면 모델 호출을 **가로채서** 조건에 따라 동작을 변경할 수 있습니다:
- `NavigatorContext.response_mode == "chat"` → `response_format=None`으로 ToolStrategy 해제
- `NavigatorContext.response_mode == "blueprint"` → ToolStrategy 유지

In [ ]:
# 📂 이 코드는 app/agents/utils.py에 있습니다.
from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

@wrap_model_call
async def dynamic_response_format(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """런타임 컨텍스트에 따라 response_format 동적 선택 (chat 모드 지원)"""
    mode = "chat"
    if request.runtime.context and hasattr(request.runtime.context, "response_mode"):
        mode = request.runtime.context.response_mode
    
    if mode == "chat":
        request = request.override(response_format=None)
    
    return await handler(request)

print("✅ dynamic_response_format 미들웨어 정의 완료")

---
### Part 8: 최종 조립 — Navigator 에이전트 완성

지금까지 이식한 모든 조각을 조립합니다:

| 구성요소 | 파트 | 실제 파일 |
|:---|:---|:---|
| 도구 3개 | Part 4 | `app/tools/navigator.py` |
| 시스템 프롬프트 | Part 5 | `app/prompts/navigator.py` |
| Blueprint 스키마 | Part 6 | `app/schemas.py` |
| 미들웨어 | Part 7 | `app/agents/utils.py` |

이 모든 것을 `create_agent()`로 조립하면 Navigator 에이전트가 완성됩니다.

> 실제 프로젝트에서는 `from app.agents.navigator import create_navigator_agent`로 한 줄 임포트합니다.

In [ ]:
# 📂 이 코드는 app/agents/navigator.py에 있습니다.
# 실제 프로젝트에서는 아래 한 줄로 임포트합니다:
# from app.agents.navigator import create_navigator_agent

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langgraph.checkpoint.memory import InMemorySaver

def create_navigator_agent(model_name: str = "google_genai:gemini-2.5-pro", temperature: float = 0.1):
    """구조화된 정보 수집 및 탐색용 에이전트 생성"""
    nav_model = init_chat_model(model_name, temperature=temperature)
    nav_checkpointer = InMemorySaver()
    
    agent = create_agent(
        model=nav_model,
        system_prompt=NAVIGATOR_SYSTEM_PROMPT,           # Part 5
        context_schema=NavigatorContext,                  # Part 6
        tools=[get_page_structure,                        # Part 4
               verify_selectors_with_samples,             # Part 4
               browse_web],                               # Part 4
        checkpointer=nav_checkpointer,
        response_format=ToolStrategy(NavigatorBlueprintCollection),  # Part 6
        middleware=[dynamic_response_format]              # Part 7
    )
    return agent

print("✅ create_navigator_agent 팩토리 함수 정의 완료")

#### 🧪 테스트 1: Blueprint 모드 — 사이트 분석

`response_mode="blueprint"`로 실행하면 Navigator가 사이트를 분석하고 **구조화된 Blueprint**를 출력합니다.
이 Blueprint는 후속 에이전트(Coder)가 크롤링 코드를 자동 생성하는 데 사용됩니다.

In [ ]:
# Navigator 에이전트 생성
navigator = create_navigator_agent()

# Blueprint 모드로 실행
config_bp = {"configurable": {"thread_id": "nav_blueprint_test"}}
context_bp = NavigatorContext(response_mode="blueprint")

response = await navigator.ainvoke(
    {"messages": [HumanMessage(
        "http://quotes.toscrape.com 에서 명언(quote), 저자(author), 태그(tags)를 수집하는 Blueprint를 만들어주세요."
    )]},
    config=config_bp,
    context=context_bp
)

# 구조화된 응답 확인
if "structured_response" in response:
    import json
    print("📋 Navigator Blueprint 출력:")
    # Pydantic 모델을 딕셔너리로 변환 (Pydantic v2 .model_dump() 또는 v1 .dict() 적용)
    bp_data = response["structured_response"].model_dump() if hasattr(response["structured_response"], "model_dump") else response["structured_response"].dict()
    print(json.dumps(bp_data, ensure_ascii=False, indent=2))
else:
    print("📝 응답:")
    print(response["messages"][-1].content)

#### 🧪 테스트 2: Chat 모드 — 자연어 대화

`response_mode="chat"`으로 실행하면 `dynamic_response_format` 미들웨어가 `ToolStrategy`를 해제하여 **자연어로 응답**합니다.
서빙 시 사용자와 일반 대화를 할 때 사용하는 모드입니다.

In [ ]:
# Chat 모드로 실행
config_chat = {"configurable": {"thread_id": "nav_chat_test"}}
context_chat = NavigatorContext(response_mode="chat")

response = await navigator.ainvoke(
    {"messages": [HumanMessage("안녕하세요! 어떤 도움을 줄 수 있나요?")]},
    config=config_chat,
    context=context_chat
)

print(f"🤖 Navigator (Chat Mode): {response['messages'][-1].content}")

---
### 📦 정리: Navigator v1의 모듈 구조

지금까지 이식한 코드가 `app/` 프로젝트에서 어떻게 모듈화되어 있는지 정리합니다.

```
app/
├── agents/
│   ├── navigator.py    ← Part 8: create_navigator_agent() 팩토리 함수
│   └── utils.py        ← Part 7: dynamic_response_format 미들웨어
├── tools/
│   ├── navigator.py    ← Part 4: get_page_structure, verify_selectors, browse_web
│   └── common.py       ← CostTracker 유틸리티
├── prompts/
│   └── navigator.py    ← Part 5: NAVIGATOR_SYSTEM_PROMPT, BROWSER_AGENT_GUIDE
└── schemas.py          ← Part 6: PageLayer, NavigatorBlueprint, NavigatorContext
```

### 🔜 다음 노트북

**04_The_Coder.ipynb**에서는 **Coder 에이전트**를 만들어, Navigator의 Blueprint를 받아 **실제 크롤링 코드를 자동 생성**합니다.

Navigator가 설계한 "청사진"을 Coder가 "코드"로 바꾸는 것 — 이것이 멀티 에이전트 파이프라인의 핵심입니다.